# PlanetScope Vegetation Index Analysis

This notebook demonstrates how to:
1. Extract spectral bands from PlanetScope (SuperDove) imagery
2. Apply cloud masking using the UDM2 quality mask
3. Compute vegetation indices (NDVI, NDRE, EVI, SAVI)
4. Visualize and export results as GeoTIFFs

**Part 2 (added):** Full crop monitoring pipeline for rice/corn:
5. Multi-date scene loading and time-series extraction over an AOI
6. Phenology curve plotting
7. Growth stage detection
8. Anomaly and stress detection
9. Sentinel-2 SWIR fusion for true NDMI

**Part 3 (added):** Planet API scene ordering from a KML file:
10. Parse KML to GeoJSON geometry
11. Search the Planet Data API for available scenes
12. Place and download orders via the Orders API

## 1. Setup and Dependencies

In [ ]:
# Install dependencies if needed
# !pip install rasterio numpy matplotlib

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import os

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 10)
plt.rcParams["figure.dpi"] = 100

## 2. Configuration

Update these paths to point to your PlanetScope scene files.

| Band | Name         | Wavelength (nm) | Key Use              |
|------|--------------|-----------------|----------------------|
| 1    | Coastal Blue | 431–452         | Atmospheric corr.    |
| 2    | Blue         | 465–515         | EVI computation      |
| 3    | Green I      | 513–549         | Green reflectance    |
| 4    | Green        | 547–583         | Chlorophyll          |
| 5    | Yellow       | 600–620         | Senescence           |
| 6    | Red          | 650–680         | Chlorophyll abs.     |
| 7    | Red Edge     | 697–713         | Canopy stress (NDRE) |
| 8    | NIR          | 845–885         | Leaf structure       |

In [ ]:
# ----- UPDATE THESE PATHS -----
SCENE_PATH = "path/to/planetscope_analytic_sr.tif"  # Analytic Surface Reflectance
UDM2_PATH  = "path/to/planetscope_udm2.tif"         # UDM2 quality mask
OUTPUT_DIR = "output"

# Scale factor: PlanetScope SR values are often 0-10000; divide to get 0-1 reflectance
SCALE_FACTOR = 10000.0

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Load and Extract Bands

In [ ]:
with rasterio.open(SCENE_PATH) as src:
    print(f"CRS      : {src.crs}")
    print(f"Size     : {src.width} x {src.height}")
    print(f"Bands    : {src.count}")
    print(f"Bounds   : {src.bounds}")
    print(f"Transform: {src.transform}")

    # Read all bands and scale to 0-1 reflectance
    coastal_blue = src.read(1).astype(float) / SCALE_FACTOR
    blue         = src.read(2).astype(float) / SCALE_FACTOR
    green_i      = src.read(3).astype(float) / SCALE_FACTOR
    green        = src.read(4).astype(float) / SCALE_FACTOR
    yellow       = src.read(5).astype(float) / SCALE_FACTOR
    red          = src.read(6).astype(float) / SCALE_FACTOR
    red_edge     = src.read(7).astype(float) / SCALE_FACTOR
    nir          = src.read(8).astype(float) / SCALE_FACTOR

    profile = src.profile.copy()

### Quick Look: RGB and False Color Composites

In [ ]:
def normalize_band(band, pmin=2, pmax=98):
    """Percentile stretch for display."""
    lo = np.nanpercentile(band, pmin)
    hi = np.nanpercentile(band, pmax)
    return np.clip((band - lo) / (hi - lo + 1e-10), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# True color: Red, Green, Blue
rgb = np.dstack([normalize_band(red), normalize_band(green), normalize_band(blue)])
axes[0].imshow(rgb)
axes[0].set_title("True Color (R-G-B)", fontsize=13)
axes[0].axis("off")

# False color: NIR, Red, Green (vegetation appears red)
nrg = np.dstack([normalize_band(nir), normalize_band(red), normalize_band(green)])
axes[1].imshow(nrg)
axes[1].set_title("False Color (NIR-R-G)", fontsize=13)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4. Cloud Masking with UDM2

The UDM2 file has 8 bands:
- Band 1: Clear (1 = usable)
- Band 2: Snow
- Band 3: Shadow
- Band 4: Light Haze
- Band 5: Heavy Haze
- Band 6: Cloud
- Band 7: Confidence
- Band 8: Unusable

In [ ]:
with rasterio.open(UDM2_PATH) as udm:
    clear_band  = udm.read(1)  # 1 = clear
    shadow_band = udm.read(3)  # 1 = shadow
    cloud_band  = udm.read(6)  # 1 = cloud
    haze_band   = udm.read(5)  # 1 = heavy haze

# Clean mask: clear, no cloud, no shadow, no heavy haze
clean_mask = (clear_band == 1) & (cloud_band == 0) & (shadow_band == 0) & (haze_band == 0)

total_pixels = clean_mask.size
clean_pixels = clean_mask.sum()
print(f"Total pixels : {total_pixels:,}")
print(f"Clean pixels : {clean_pixels:,} ({clean_pixels/total_pixels*100:.1f}%)")
print(f"Masked pixels: {total_pixels - clean_pixels:,} ({(total_pixels - clean_pixels)/total_pixels*100:.1f}%)")

# Visualize the mask
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(clean_mask, cmap="gray")
ax.set_title("UDM2 Clean Mask (white = usable)", fontsize=13)
ax.axis("off")
plt.show()

## 5. Compute Vegetation Indices

In [ ]:
# Small epsilon to avoid division by zero
EPS = 1e-10

# --- NDVI: Normalized Difference Vegetation Index ---
# Standard measure of vegetation greenness and density
ndvi = (nir - red) / (nir + red + EPS)

# --- NDRE: Normalized Difference Red Edge ---
# More sensitive to chlorophyll in dense/mature canopies
# Key advantage of PlanetScope's Red Edge band
ndre = (nir - red_edge) / (nir + red_edge + EPS)

# --- EVI: Enhanced Vegetation Index ---
# Reduces atmospheric and soil noise; better for high-biomass areas
evi = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0)

# --- SAVI: Soil-Adjusted Vegetation Index ---
# Minimizes soil brightness influence; useful for sparse vegetation
L = 0.5  # soil brightness correction factor
savi = ((nir - red) / (nir + red + L)) * (1.0 + L)

# Apply cloud mask to all indices
ndvi_masked = np.where(clean_mask, ndvi, np.nan)
ndre_masked = np.where(clean_mask, ndre, np.nan)
evi_masked  = np.where(clean_mask, evi, np.nan)
savi_masked = np.where(clean_mask, savi, np.nan)

print("Vegetation indices computed and cloud-masked.")

## 6. Visualize Vegetation Indices

In [ ]:
indices = {
    "NDVI": {"data": ndvi_masked, "vmin": -0.2, "vmax": 0.8, "cmap": "RdYlGn"},
    "NDRE": {"data": ndre_masked, "vmin": -0.1, "vmax": 0.5, "cmap": "RdYlGn"},
    "EVI":  {"data": evi_masked,  "vmin": -0.2, "vmax": 0.8, "cmap": "RdYlGn"},
    "SAVI": {"data": savi_masked, "vmin": -0.2, "vmax": 0.8, "cmap": "RdYlGn"},
}

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, (name, cfg) in zip(axes.flat, indices.items()):
    im = ax.imshow(cfg["data"], cmap=cfg["cmap"], vmin=cfg["vmin"], vmax=cfg["vmax"])
    ax.set_title(name, fontsize=14, fontweight="bold")
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.75, label=name)

plt.suptitle("PlanetScope Vegetation Indices (Cloud-Masked)", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 7. Statistical Summary

In [ ]:
print(f"{'Index':<8} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8} {'Median':>8}")
print("-" * 52)

for name, cfg in indices.items():
    data = cfg["data"]
    valid = data[~np.isnan(data)]
    print(f"{name:<8} {valid.min():>8.4f} {valid.max():>8.4f} {valid.mean():>8.4f} {valid.std():>8.4f} {np.median(valid):>8.4f}")

## 8. NDVI Histogram and Classification

In [ ]:
ndvi_valid = ndvi_masked[~np.isnan(ndvi_masked)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(ndvi_valid, bins=100, color="forestgreen", alpha=0.8, edgecolor="darkgreen")
axes[0].axvline(x=0.2, color="orange", linestyle="--", label="Sparse veg. (0.2)")
axes[0].axvline(x=0.5, color="red", linestyle="--", label="Dense veg. (0.5)")
axes[0].set_xlabel("NDVI", fontsize=12)
axes[0].set_ylabel("Pixel Count", fontsize=12)
axes[0].set_title("NDVI Distribution", fontsize=13)
axes[0].legend()

# Classification map
ndvi_classes = np.full_like(ndvi_masked, np.nan)
ndvi_classes[ndvi_masked < 0.0]  = 0  # Water / Non-vegetation
ndvi_classes[(ndvi_masked >= 0.0) & (ndvi_masked < 0.2)] = 1  # Bare soil
ndvi_classes[(ndvi_masked >= 0.2) & (ndvi_masked < 0.4)] = 2  # Sparse vegetation
ndvi_classes[(ndvi_masked >= 0.4) & (ndvi_masked < 0.6)] = 3  # Moderate vegetation
ndvi_classes[ndvi_masked >= 0.6] = 4  # Dense vegetation

from matplotlib.colors import ListedColormap
class_cmap = ListedColormap(["#2166ac", "#d6a04e", "#c2e699", "#4dac26", "#006837"])
class_labels = ["Water/Non-veg", "Bare Soil", "Sparse Veg", "Moderate Veg", "Dense Veg"]

im = axes[1].imshow(ndvi_classes, cmap=class_cmap, vmin=0, vmax=4)
axes[1].set_title("NDVI Land Cover Classification", fontsize=13)
axes[1].axis("off")
cbar = plt.colorbar(im, ax=axes[1], ticks=[0, 1, 2, 3, 4], shrink=0.8)
cbar.ax.set_yticklabels(class_labels)

plt.tight_layout()
plt.show()

## 9. NDVI vs NDRE Comparison

NDRE leverages the Red Edge band (unique to PlanetScope) and is more sensitive to chlorophyll variation in dense canopies where NDVI saturates.

In [ ]:
# Scatter plot of NDVI vs NDRE
fig, ax = plt.subplots(figsize=(8, 8))

# Sample pixels for performance
mask_flat = clean_mask.flatten()
ndvi_flat = ndvi.flatten()[mask_flat]
ndre_flat = ndre.flatten()[mask_flat]

sample_size = min(50000, len(ndvi_flat))
rng = np.random.default_rng(42)
idx = rng.choice(len(ndvi_flat), size=sample_size, replace=False)

ax.scatter(ndvi_flat[idx], ndre_flat[idx], s=1, alpha=0.3, c="forestgreen")
ax.set_xlabel("NDVI", fontsize=12)
ax.set_ylabel("NDRE", fontsize=12)
ax.set_title("NDVI vs NDRE (sampled pixels)", fontsize=13)
ax.set_xlim(-0.3, 1.0)
ax.set_ylim(-0.2, 0.6)
ax.axhline(y=0, color="gray", linewidth=0.5)
ax.axvline(x=0, color="gray", linewidth=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: In dense canopy areas (high NDVI), NDRE still shows variation,")
print("demonstrating its sensitivity to chlorophyll content beyond NDVI saturation.")

## 10. Export Results as GeoTIFFs

In [ ]:
def export_index(data, name, output_dir, profile):
    """Export a vegetation index as a single-band GeoTIFF."""
    out_profile = profile.copy()
    out_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

    output_path = os.path.join(output_dir, f"{name.lower()}.tif")
    with rasterio.open(output_path, "w", **out_profile) as dst:
        dst.write(data.astype(np.float32), 1)
    print(f"Exported: {output_path}")
    return output_path

# Export all indices
for name, cfg in indices.items():
    export_index(cfg["data"], name, OUTPUT_DIR, profile)

print(f"\nAll indices exported to: {os.path.abspath(OUTPUT_DIR)}")

## 11. Band Spectral Profile (Optional)

Plot the average spectral signature of vegetated vs non-vegetated pixels.

In [ ]:
# Define vegetation mask from NDVI
veg_mask = clean_mask & (ndvi > 0.4)
nonveg_mask = clean_mask & (ndvi < 0.1) & (ndvi > -0.1)

band_names = ["Coastal\nBlue", "Blue", "Green I", "Green", "Yellow", "Red", "Red\nEdge", "NIR"]
wavelengths = [442, 490, 531, 565, 610, 665, 705, 865]  # center wavelengths (nm)
all_bands = [coastal_blue, blue, green_i, green, yellow, red, red_edge, nir]

veg_means = [np.nanmean(b[veg_mask]) for b in all_bands]
nonveg_means = [np.nanmean(b[nonveg_mask]) for b in all_bands]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(wavelengths, veg_means, "o-", color="forestgreen", linewidth=2, markersize=8, label="Vegetation (NDVI > 0.4)")
ax.plot(wavelengths, nonveg_means, "s--", color="sienna", linewidth=2, markersize=8, label="Non-vegetation (NDVI ~ 0)")

ax.set_xlabel("Wavelength (nm)", fontsize=12)
ax.set_ylabel("Surface Reflectance", fontsize=12)
ax.set_title("PlanetScope Spectral Profiles", fontsize=14)
ax.set_xticks(wavelengths)
ax.set_xticklabels(band_names, fontsize=10)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Highlight the red edge region
ax.axvspan(690, 720, alpha=0.1, color="red", label="Red Edge region")

plt.tight_layout()
plt.show()

print("The vegetation curve shows the classic red-edge jump between Red (665nm) and NIR (865nm).")
print("PlanetScope's Red Edge band (705nm) captures the inflection point of this transition.")

---

## Summary (Part 1)

| Index | Formula | Best For |
|-------|---------|----------|
| **NDVI** | (NIR - Red) / (NIR + Red) | General vegetation mapping |
| **NDRE** | (NIR - RedEdge) / (NIR + RedEdge) | Chlorophyll in dense canopy |
| **EVI** | 2.5 x (NIR - Red) / (NIR + 6xRed - 7.5xBlue + 1) | High biomass / tropical |
| **SAVI** | ((NIR - Red) / (NIR + Red + L)) x (1+L) | Sparse / arid vegetation |

### Key Takeaways
- Always use **Surface Reflectance** products and apply the **UDM2 cloud mask**
- **NDRE** is uniquely available from PlanetScope and outperforms NDVI in dense canopies
- Scale PlanetScope SR values by dividing by 10000 before computing indices
- Export results as GeoTIFFs to preserve georeferencing for GIS workflows

---

# Part 2: Crop Growth Monitoring Pipeline (Rice / Corn)

This section builds a full **sow-to-harvest** monitoring pipeline using multi-date PlanetScope imagery.

**What we cover:**
- Loading a time-series of PlanetScope scenes over a field AOI
- Computing NDVI, NDRE, EVI, SAVI for each date (with cloud masking)
- Building and plotting phenology curves
- Automatically detecting growth stages from index trajectories
- Identifying stress and anomaly events
- Fusing Sentinel-2 SWIR for true NDMI (moisture index)

### Crop Season Reference

| Crop | Season | Key Stages |
|------|--------|------------|
| Rice | 120-150 days | Transplanting > Tillering > Heading > Grain fill > Harvest |
| Corn | 90-130 days | Emergence > V6 (knee-high) > Tasseling > Grain fill > Maturity |

## 12. Multi-Date Configuration

In [ ]:
import glob
import pandas as pd
from datetime import datetime
import json
import warnings
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# ----- UPDATE THESE PATHS -----

# Directory containing time-series PlanetScope scenes.
# Expected structure (Planet SDK default download layout):
#   SCENES_DIR/
#     20240301_021345_67_2489_3B_AnalyticMS_SR_8b.tif
#     20240301_021345_67_2489_3B_udm2.tif
#     20240308_021012_34_1056_3B_AnalyticMS_SR_8b.tif
#     20240308_021012_34_1056_3B_udm2.tif
#     ...
SCENES_DIR = "path/to/scenes_directory"

# AOI: GeoJSON file defining the field boundary, or None to use full scene
AOI_GEOJSON = "path/to/field_boundary.geojson"  # set to None to skip clipping

# Crop type for phenology reference
CROP_TYPE = "rice"  # "rice" or "corn"

# Growing season date range (inclusive)
SEASON_START = "2024-03-01"
SEASON_END   = "2024-08-15"

# Minimum percentage of clean (cloud-free) pixels to accept a scene
MIN_CLEAN_PCT = 70.0

SCALE_FACTOR = 10000.0
EPS = 1e-10

OUTPUT_DIR_TS = "output/timeseries"
os.makedirs(OUTPUT_DIR_TS, exist_ok=True)

print(f"Crop: {CROP_TYPE}")
print(f"Season: {SEASON_START} to {SEASON_END}")
print(f"Scenes directory: {SCENES_DIR}")

## 13. Discover and Pair Scene Files

Automatically find all SR + UDM2 file pairs and extract dates from filenames.

In [ ]:
import re

def discover_scene_pairs(scenes_dir, date_start, date_end):
    """Find matching SR and UDM2 file pairs within a date range."""
    sr_files = sorted(glob.glob(os.path.join(scenes_dir, "*SR*8b*.tif")) +
                      glob.glob(os.path.join(scenes_dir, "*AnalyticMS_SR*.tif")))
    
    pairs = []
    date_pattern = re.compile(r"(\d{8})_")
    start_dt = datetime.strptime(date_start, "%Y-%m-%d")
    end_dt = datetime.strptime(date_end, "%Y-%m-%d")
    
    for sr_path in sr_files:
        match = date_pattern.search(os.path.basename(sr_path))
        if not match:
            continue
        scene_date = datetime.strptime(match.group(1), "%Y%m%d")
        if not (start_dt <= scene_date <= end_dt):
            continue
        
        # Find corresponding UDM2 file
        base = os.path.basename(sr_path)
        scene_id = "_".join(base.split("_")[:4])  # e.g., 20240301_021345_67_2489
        udm2_candidates = glob.glob(os.path.join(scenes_dir, f"{scene_id}*udm2*.tif"))
        
        udm2_path = udm2_candidates[0] if udm2_candidates else None
        pairs.append({
            "date": scene_date,
            "date_str": scene_date.strftime("%Y-%m-%d"),
            "sr_path": sr_path,
            "udm2_path": udm2_path,
        })
    
    pairs.sort(key=lambda x: x["date"])
    return pairs

scene_pairs = discover_scene_pairs(SCENES_DIR, SEASON_START, SEASON_END)
print(f"Found {len(scene_pairs)} scenes in date range:")
for p in scene_pairs:
    udm_status = "+ UDM2" if p["udm2_path"] else "(no UDM2)"
    print(f"  {p['date_str']}  {udm_status}  {os.path.basename(p['sr_path'])}")

## 14. AOI Loading and Clipping Helper

In [ ]:
from rasterio.mask import mask as rio_mask
from shapely.geometry import shape

def load_aoi(geojson_path):
    """Load AOI geometry from a GeoJSON file."""
    if geojson_path is None or not os.path.exists(geojson_path):
        return None
    with open(geojson_path) as f:
        gj = json.load(f)
    # Handle both FeatureCollection and single Feature/Geometry
    if gj["type"] == "FeatureCollection":
        geoms = [shape(feat["geometry"]) for feat in gj["features"]]
    elif gj["type"] == "Feature":
        geoms = [shape(gj["geometry"])]
    else:
        geoms = [shape(gj)]
    return geoms

def read_bands_clipped(sr_path, aoi_geoms=None, scale_factor=10000.0):
    """Read 8-band PlanetScope SR, optionally clipped to AOI. Returns dict of bands + profile."""
    with rasterio.open(sr_path) as src:
        if aoi_geoms is not None:
            out_image, out_transform = rio_mask(src, aoi_geoms, crop=True, nodata=0)
            profile = src.profile.copy()
            profile.update(transform=out_transform,
                           width=out_image.shape[2],
                           height=out_image.shape[1])
        else:
            out_image = src.read()
            profile = src.profile.copy()
        
        bands = {}
        band_names = ["coastal_blue", "blue", "green_i", "green",
                      "yellow", "red", "red_edge", "nir"]
        for i, name in enumerate(band_names):
            if i < out_image.shape[0]:
                bands[name] = out_image[i].astype(float) / scale_factor
        
        return bands, profile

def read_udm2_clipped(udm2_path, aoi_geoms=None):
    """Read UDM2 mask, optionally clipped to AOI. Returns clean_mask boolean array."""
    if udm2_path is None or not os.path.exists(udm2_path):
        return None
    with rasterio.open(udm2_path) as src:
        if aoi_geoms is not None:
            out_image, _ = rio_mask(src, aoi_geoms, crop=True, nodata=0)
        else:
            out_image = src.read()
        clear  = out_image[0]  # Band 1: clear
        shadow = out_image[2]  # Band 3: shadow
        haze   = out_image[4]  # Band 5: heavy haze
        cloud  = out_image[5]  # Band 6: cloud
    return (clear == 1) & (cloud == 0) & (shadow == 0) & (haze == 0)

aoi_geoms = load_aoi(AOI_GEOJSON)
if aoi_geoms:
    print(f"AOI loaded: {len(aoi_geoms)} geometry(ies)")
else:
    print("No AOI specified -- will use full scene extent.")

## 15. Process All Scenes: Extract Time-Series Indices Over AOI

In [ ]:
def compute_indices(bands):
    """Compute all vegetation indices from a band dictionary."""
    nir = bands["nir"]
    red = bands["red"]
    red_edge = bands["red_edge"]
    blue = bands["blue"]
    
    ndvi = (nir - red) / (nir + red + EPS)
    ndre = (nir - red_edge) / (nir + red_edge + EPS)
    evi  = 2.5 * (nir - red) / (nir + 6.0 * red - 7.5 * blue + 1.0)
    L = 0.5
    savi = ((nir - red) / (nir + red + L)) * (1.0 + L)
    
    return {"ndvi": ndvi, "ndre": ndre, "evi": evi, "savi": savi}

# Process each scene
timeseries_records = []

for pair in scene_pairs:
    bands, prof = read_bands_clipped(pair["sr_path"], aoi_geoms, SCALE_FACTOR)
    clean_mask = read_udm2_clipped(pair["udm2_path"], aoi_geoms)
    
    # If no UDM2, assume all pixels are clean
    if clean_mask is None:
        clean_mask = np.ones_like(bands["nir"], dtype=bool)
    
    # Check cloud-free percentage
    total_px = clean_mask.size
    clean_px = clean_mask.sum()
    clean_pct = (clean_px / total_px) * 100 if total_px > 0 else 0
    
    if clean_pct < MIN_CLEAN_PCT:
        print(f"  {pair['date_str']}  SKIPPED  ({clean_pct:.0f}% clean < {MIN_CLEAN_PCT}% threshold)")
        continue
    
    idx = compute_indices(bands)
    
    # Compute field-level statistics (mean, std, percentiles) over clean pixels
    record = {"date": pair["date"], "date_str": pair["date_str"], "clean_pct": clean_pct}
    for name, arr in idx.items():
        valid = arr[clean_mask]
        record[f"{name}_mean"] = np.nanmean(valid)
        record[f"{name}_std"]  = np.nanstd(valid)
        record[f"{name}_p10"]  = np.nanpercentile(valid, 10)
        record[f"{name}_p50"]  = np.nanpercentile(valid, 50)
        record[f"{name}_p90"]  = np.nanpercentile(valid, 90)
    
    timeseries_records.append(record)
    print(f"  {pair['date_str']}  OK  ({clean_pct:.0f}% clean)  NDVI={record['ndvi_mean']:.3f}  NDRE={record['ndre_mean']:.3f}")

# Build DataFrame
ts_df = pd.DataFrame(timeseries_records)
ts_df["date"] = pd.to_datetime(ts_df["date"])
ts_df = ts_df.sort_values("date").reset_index(drop=True)

# Days since sowing (first observation)
ts_df["days_since_start"] = (ts_df["date"] - ts_df["date"].iloc[0]).dt.days

print(f"\nTime-series built: {len(ts_df)} usable observations")
ts_df[["date_str", "clean_pct", "ndvi_mean", "ndre_mean", "evi_mean", "savi_mean"]].head(10)

## 16. Plot Phenology Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=True)

index_configs = [
    ("ndvi", "NDVI", "forestgreen", (-0.1, 1.0)),
    ("ndre", "NDRE", "teal", (-0.1, 0.6)),
    ("evi",  "EVI",  "darkorange", (-0.1, 1.0)),
    ("savi", "SAVI", "saddlebrown", (-0.1, 1.0)),
]

for ax, (key, label, color, ylim) in zip(axes.flat, index_configs):
    # Mean line
    ax.plot(ts_df["date"], ts_df[f"{key}_mean"], "o-", color=color,
            linewidth=2, markersize=5, label=f"{label} (mean)")
    # Interquartile shading (p10 to p90)
    ax.fill_between(ts_df["date"], ts_df[f"{key}_p10"], ts_df[f"{key}_p90"],
                    alpha=0.2, color=color, label="P10-P90 range")
    ax.set_ylabel(label, fontsize=12)
    ax.set_ylim(ylim)
    ax.legend(loc="upper left", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis="x", rotation=45)

fig.suptitle(f"Crop Phenology Curves ({CROP_TYPE.title()}) - PlanetScope Time-Series",
             fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### Combined Phenology: NDVI vs NDRE

Plotting NDVI and NDRE together highlights how NDRE continues to show variation in dense canopy where NDVI saturates.

In [ ]:
fig, ax1 = plt.subplots(figsize=(16, 6))

# NDVI on left axis
ln1 = ax1.plot(ts_df["date"], ts_df["ndvi_mean"], "o-", color="forestgreen",
               linewidth=2, markersize=5, label="NDVI")
ax1.fill_between(ts_df["date"], ts_df["ndvi_p10"], ts_df["ndvi_p90"],
                 alpha=0.15, color="forestgreen")
ax1.set_ylabel("NDVI", fontsize=12, color="forestgreen")
ax1.set_ylim(-0.1, 1.0)
ax1.tick_params(axis="y", labelcolor="forestgreen")

# NDRE on right axis
ax2 = ax1.twinx()
ln2 = ax2.plot(ts_df["date"], ts_df["ndre_mean"], "s--", color="teal",
               linewidth=2, markersize=5, label="NDRE")
ax2.fill_between(ts_df["date"], ts_df["ndre_p10"], ts_df["ndre_p90"],
                 alpha=0.15, color="teal")
ax2.set_ylabel("NDRE", fontsize=12, color="teal")
ax2.set_ylim(-0.1, 0.6)
ax2.tick_params(axis="y", labelcolor="teal")

# Combined legend
lns = ln1 + ln2
labs = [l.get_label() for l in lns]
ax1.legend(lns, labs, loc="upper left", fontsize=11)

ax1.set_title(f"NDVI vs NDRE Phenology ({CROP_TYPE.title()})", fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("When NDVI plateaus (saturates) at peak canopy, NDRE continues to differentiate")
print("chlorophyll concentration -- making it better for mid-season health assessment.")

## 17. Automatic Growth Stage Detection

We classify each observation into a growth stage based on NDVI trajectory:

| Signal | Stage |
|--------|-------|
| NDVI < 0.2, flat | Bare soil / Pre-emergence |
| NDVI rising (derivative > 0) | Vegetative growth |
| NDVI plateau > 0.65 | Peak canopy |
| NDVI declining (derivative < 0, NDVI > 0.3) | Reproductive / Grain fill |
| NDVI < 0.3, declining | Senescence / Near harvest |

In [ ]:
def detect_growth_stages(df, ndvi_col="ndvi_mean"):
    """Classify each date into a crop growth stage based on NDVI trajectory."""
    stages = []
    ndvi_vals = df[ndvi_col].values
    
    # Compute smoothed first derivative (rate of change)
    # Use centered differences; pad edges with forward/backward diff
    deriv = np.gradient(ndvi_vals)
    
    # Track whether we've reached peak (to distinguish rising vs falling)
    peak_ndvi = ndvi_vals.max()
    peak_idx = ndvi_vals.argmax()
    
    for i, (ndvi_val, d) in enumerate(zip(ndvi_vals, deriv)):
        if ndvi_val < 0.15:
            stage = "Bare Soil / Pre-emergence"
        elif ndvi_val < 0.3 and i < peak_idx:
            stage = "Early Emergence"
        elif d > 0.005 and i <= peak_idx:
            stage = "Vegetative Growth"
        elif ndvi_val >= 0.65 and abs(d) <= 0.01:
            stage = "Peak Canopy"
        elif d < -0.005 and ndvi_val > 0.3:
            stage = "Reproductive / Grain Fill"
        elif ndvi_val <= 0.3 and i > peak_idx:
            stage = "Senescence / Harvest"
        else:
            stage = "Transition"
        stages.append(stage)
    
    return stages

ts_df["growth_stage"] = detect_growth_stages(ts_df)

# Display
print(f"{'Date':<14} {'NDVI':>6} {'NDRE':>6} {'Stage'}")
print("-" * 58)
for _, row in ts_df.iterrows():
    print(f"{row['date_str']:<14} {row['ndvi_mean']:>6.3f} {row['ndre_mean']:>6.3f} {row['growth_stage']}")

In [ ]:
# Visualize growth stages on the NDVI curve
stage_colors = {
    "Bare Soil / Pre-emergence": "#d6a04e",
    "Early Emergence": "#c2e699",
    "Vegetative Growth": "#4dac26",
    "Peak Canopy": "#006837",
    "Reproductive / Grain Fill": "#fd8d3c",
    "Senescence / Harvest": "#bd0026",
    "Transition": "#999999",
}

fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(ts_df["date"], ts_df["ndvi_mean"], "-", color="gray", linewidth=1, alpha=0.5)

for stage, color in stage_colors.items():
    mask = ts_df["growth_stage"] == stage
    if mask.any():
        ax.scatter(ts_df.loc[mask, "date"], ts_df.loc[mask, "ndvi_mean"],
                   c=color, s=80, label=stage, edgecolors="black", linewidth=0.5, zorder=5)

ax.set_ylabel("NDVI (field mean)", fontsize=12)
ax.set_title(f"Growth Stage Detection ({CROP_TYPE.title()})", fontsize=14)
ax.legend(loc="upper left", fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 18. Stress and Anomaly Detection

Identify dates where the crop shows signs of stress by comparing observed values against expected phenology.

| Condition | Detection Signal |
|-----------|------------------|
| Water stress | NDRE drops while NDVI stays high (early indicator) |
| Nutrient deficiency | Lower NDVI plateau than healthy reference |
| Pest / disease | Localized NDVI drop within the field |
| Lodging | Sudden mid-season NDVI drop |

In [ ]:
def detect_anomalies(df):
    """Flag anomalous observations based on index behavior."""
    alerts = []
    ndvi = df["ndvi_mean"].values
    ndre = df["ndre_mean"].values
    std  = df["ndvi_std"].values
    dates = df["date_str"].values
    
    for i in range(1, len(ndvi)):
        ndvi_delta = ndvi[i] - ndvi[i - 1]
        ndre_delta = ndre[i] - ndre[i - 1]
        
        # Sudden NDVI drop (possible lodging, pest, or weather damage)
        if ndvi_delta < -0.10 and ndvi[i - 1] > 0.4:
            alerts.append({
                "date": dates[i],
                "type": "Sudden NDVI Drop",
                "severity": "HIGH",
                "detail": f"NDVI dropped {ndvi_delta:+.3f} (from {ndvi[i-1]:.3f} to {ndvi[i]:.3f}). "
                          f"Possible lodging, pest damage, or severe weather event."
            })
        
        # Early water/chlorophyll stress: NDRE drops while NDVI stable
        if ndre_delta < -0.05 and abs(ndvi_delta) < 0.03 and ndvi[i] > 0.5:
            alerts.append({
                "date": dates[i],
                "type": "Early Stress (NDRE)",
                "severity": "MEDIUM",
                "detail": f"NDRE dropped {ndre_delta:+.3f} while NDVI stable ({ndvi_delta:+.3f}). "
                          f"Possible water stress or chlorophyll decline before visible canopy change."
            })
        
        # High spatial variability (intra-field heterogeneity)
        if std[i] > 0.15 and ndvi[i] > 0.3:
            alerts.append({
                "date": dates[i],
                "type": "High Intra-Field Variability",
                "severity": "LOW",
                "detail": f"NDVI std={std[i]:.3f} indicates uneven growth. "
                          f"Check for patchy pest/disease, irrigation issues, or soil variability."
            })
        
        # Stalled growth: expected rise but NDVI flat during early/mid season
        peak_idx = ndvi.argmax()
        if i < peak_idx and abs(ndvi_delta) < 0.01 and ndvi[i] > 0.2 and ndvi[i] < 0.5:
            alerts.append({
                "date": dates[i],
                "type": "Stalled Growth",
                "severity": "MEDIUM",
                "detail": f"NDVI flat ({ndvi_delta:+.3f}) during expected vegetative phase. "
                          f"Possible nutrient deficiency or water limitation."
            })
    
    return alerts

alerts = detect_anomalies(ts_df)

if alerts:
    print(f"Found {len(alerts)} alert(s):\n")
    for a in alerts:
        print(f"  [{a['severity']}] {a['date']} - {a['type']}")
        print(f"    {a['detail']}\n")
else:
    print("No anomalies detected -- crop appears healthy throughout the season.")

In [ ]:
# Visualize alerts on the phenology curve
fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(ts_df["date"], ts_df["ndvi_mean"], "o-", color="forestgreen",
        linewidth=2, markersize=5, label="NDVI (mean)")
ax.fill_between(ts_df["date"], ts_df["ndvi_p10"], ts_df["ndvi_p90"],
                alpha=0.15, color="forestgreen", label="P10-P90")

severity_markers = {"HIGH": ("v", 150, "red"), "MEDIUM": ("D", 100, "orange"), "LOW": ("s", 80, "gold")}

for a in alerts:
    row = ts_df[ts_df["date_str"] == a["date"]]
    if row.empty:
        continue
    marker, size, color = severity_markers.get(a["severity"], ("o", 80, "gray"))
    ax.scatter(row["date"].values[0], row["ndvi_mean"].values[0],
              marker=marker, s=size, c=color, edgecolors="black",
              linewidth=1, zorder=10)
    ax.annotate(a["type"], xy=(row["date"].values[0], row["ndvi_mean"].values[0]),
               xytext=(10, 10), textcoords="offset points", fontsize=8,
               color=color, fontweight="bold",
               arrowprops=dict(arrowstyle="->", color=color, lw=0.8))

# Legend for severity
for sev, (m, s, c) in severity_markers.items():
    ax.scatter([], [], marker=m, s=s, c=c, edgecolors="black", linewidth=1, label=f"{sev} alert")

ax.set_ylabel("NDVI", fontsize=12)
ax.set_title(f"Stress & Anomaly Alerts ({CROP_TYPE.title()})", fontsize=14)
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 19. Sentinel-2 SWIR Fusion for True NDMI

PlanetScope lacks a SWIR band, so we fuse with Sentinel-2 Band 11 (1610 nm) to compute the true **NDMI (Normalized Difference Moisture Index)**.

**NDMI = (NIR - SWIR) / (NIR + SWIR)**

Approach:
1. Download Sentinel-2 L2A scenes covering the same dates and AOI
2. Resample S2 SWIR (20m) to match PlanetScope grid (3m) using bilinear interpolation
3. Combine PlanetScope NIR (3m) with resampled S2 SWIR to compute high-resolution NDMI

In [ ]:
from rasterio.warp import reproject, Resampling

def compute_ndmi_fused(ps_nir_path, s2_swir_path, aoi_geoms=None, ps_scale=10000.0):
    """
    Compute NDMI by fusing PlanetScope NIR (3m) with Sentinel-2 SWIR (20m).
    
    Parameters
    ----------
    ps_nir_path : str
        Path to PlanetScope 8-band SR GeoTIFF (Band 8 = NIR).
    s2_swir_path : str
        Path to Sentinel-2 L2A Band 11 (SWIR 1610nm) GeoTIFF.
    aoi_geoms : list or None
        AOI geometries for clipping.
    ps_scale : float
        Scale factor for PlanetScope SR values.
    
    Returns
    -------
    ndmi : np.ndarray
        NDMI array at PlanetScope resolution.
    profile : dict
        Rasterio profile for the output.
    """
    # Read PlanetScope NIR
    with rasterio.open(ps_nir_path) as ps_src:
        if aoi_geoms:
            ps_data, ps_transform = rio_mask(ps_src, aoi_geoms, crop=True, nodata=0)
            ps_profile = ps_src.profile.copy()
            ps_profile.update(transform=ps_transform,
                              width=ps_data.shape[2],
                              height=ps_data.shape[1])
        else:
            ps_data = ps_src.read()
            ps_profile = ps_src.profile.copy()
            ps_transform = ps_src.transform
        
        nir = ps_data[7].astype(float) / ps_scale  # Band 8 (0-indexed: 7)
    
    # Read and reproject Sentinel-2 SWIR to PlanetScope grid
    with rasterio.open(s2_swir_path) as s2_src:
        swir_reprojected = np.empty_like(nir)
        reproject(
            source=rasterio.band(s2_src, 1),
            destination=swir_reprojected,
            src_transform=s2_src.transform,
            src_crs=s2_src.crs,
            dst_transform=ps_transform,
            dst_crs=ps_profile["crs"],
            resampling=Resampling.bilinear,
        )
    
    # Sentinel-2 L2A reflectance is scaled by 10000
    swir = swir_reprojected.astype(float) / 10000.0
    
    # Compute NDMI
    ndmi = (nir - swir) / (nir + swir + EPS)
    
    out_profile = ps_profile.copy()
    out_profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)
    
    return ndmi, out_profile

print("NDMI fusion function ready.")
print("Usage:")
print('  ndmi, prof = compute_ndmi_fused("ps_scene.tif", "S2_B11.tif", aoi_geoms)')

In [ ]:
# ----- UPDATE THESE PATHS TO YOUR SENTINEL-2 FILES -----
# Map of date -> Sentinel-2 B11 file path
# Dates should be close matches (within ~2-3 days) to PlanetScope scenes
S2_SWIR_FILES = {
    # "2024-03-10": "path/to/S2A_MSIL2A_20240310_B11.tif",
    # "2024-03-20": "path/to/S2A_MSIL2A_20240320_B11.tif",
    # "2024-04-01": "path/to/S2A_MSIL2A_20240401_B11.tif",
}

# Compute NDMI time-series (only for dates with S2 data)
ndmi_records = []

for pair in scene_pairs:
    s2_path = S2_SWIR_FILES.get(pair["date_str"])
    if s2_path is None or not os.path.exists(s2_path):
        continue
    
    ndmi, ndmi_prof = compute_ndmi_fused(pair["sr_path"], s2_path, aoi_geoms)
    clean_mask_date = read_udm2_clipped(pair["udm2_path"], aoi_geoms)
    if clean_mask_date is None:
        clean_mask_date = np.ones_like(ndmi, dtype=bool)
    
    ndmi_masked = np.where(clean_mask_date, ndmi, np.nan)
    valid = ndmi_masked[~np.isnan(ndmi_masked)]
    
    ndmi_records.append({
        "date": pair["date"],
        "date_str": pair["date_str"],
        "ndmi_mean": np.nanmean(valid),
        "ndmi_std": np.nanstd(valid),
        "ndmi_p10": np.nanpercentile(valid, 10),
        "ndmi_p90": np.nanpercentile(valid, 90),
    })
    print(f"  {pair['date_str']}  NDMI={np.nanmean(valid):.3f}")

if ndmi_records:
    ndmi_df = pd.DataFrame(ndmi_records)
    ndmi_df["date"] = pd.to_datetime(ndmi_df["date"])
    
    # Merge into main time-series
    ts_df = ts_df.merge(ndmi_df[["date_str", "ndmi_mean", "ndmi_std", "ndmi_p10", "ndmi_p90"]],
                        on="date_str", how="left")
    print(f"\nNDMI added for {len(ndmi_records)} dates.")
else:
    print("No Sentinel-2 SWIR files configured. Update S2_SWIR_FILES above to enable NDMI fusion.")
    print("In the meantime, NDRE serves as a partial proxy for canopy moisture sensitivity.")

## 20. Export Time-Series Data

In [ ]:
# Save time-series as CSV
csv_path = os.path.join(OUTPUT_DIR_TS, f"{CROP_TYPE}_phenology_timeseries.csv")
ts_df.to_csv(csv_path, index=False)
print(f"Time-series CSV saved: {csv_path}")

# Save alerts as JSON
if alerts:
    alerts_path = os.path.join(OUTPUT_DIR_TS, f"{CROP_TYPE}_stress_alerts.json")
    with open(alerts_path, "w") as f:
        json.dump(alerts, f, indent=2)
    print(f"Alerts JSON saved: {alerts_path}")

print(f"\nAll outputs in: {os.path.abspath(OUTPUT_DIR_TS)}")

## 21. Full Season Dashboard

A combined view showing all indices, growth stages, and alerts.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(18, 20), sharex=True)

# Panel 1: NDVI with growth stages
ax = axes[0]
for stage, color in stage_colors.items():
    mask = ts_df["growth_stage"] == stage
    if mask.any():
        ax.scatter(ts_df.loc[mask, "date"], ts_df.loc[mask, "ndvi_mean"],
                   c=color, s=60, label=stage, edgecolors="black", linewidth=0.5, zorder=5)
ax.plot(ts_df["date"], ts_df["ndvi_mean"], "-", color="gray", linewidth=1, alpha=0.5)
ax.fill_between(ts_df["date"], ts_df["ndvi_p10"], ts_df["ndvi_p90"], alpha=0.1, color="green")
ax.set_ylabel("NDVI", fontsize=12)
ax.set_title(f"Full Season Dashboard: {CROP_TYPE.title()}", fontsize=16)
ax.legend(loc="upper left", fontsize=8, ncol=3)
ax.grid(True, alpha=0.3)

# Panel 2: NDRE (chlorophyll sensitivity)
ax = axes[1]
ax.plot(ts_df["date"], ts_df["ndre_mean"], "o-", color="teal", linewidth=2, markersize=4)
ax.fill_between(ts_df["date"], ts_df["ndre_p10"], ts_df["ndre_p90"], alpha=0.15, color="teal")
ax.set_ylabel("NDRE", fontsize=12)
ax.grid(True, alpha=0.3)

# Panel 3: EVI + SAVI
ax = axes[2]
ax.plot(ts_df["date"], ts_df["evi_mean"], "o-", color="darkorange", linewidth=2, markersize=4, label="EVI")
ax.plot(ts_df["date"], ts_df["savi_mean"], "s--", color="saddlebrown", linewidth=2, markersize=4, label="SAVI")
ax.set_ylabel("EVI / SAVI", fontsize=12)
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)

# Panel 4: NDMI (if available) or NDRE as proxy
ax = axes[3]
if "ndmi_mean" in ts_df.columns and ts_df["ndmi_mean"].notna().any():
    ndmi_valid = ts_df.dropna(subset=["ndmi_mean"])
    ax.plot(ndmi_valid["date"], ndmi_valid["ndmi_mean"], "o-", color="steelblue",
            linewidth=2, markersize=5, label="NDMI (S2 SWIR fusion)")
    ax.fill_between(ndmi_valid["date"], ndmi_valid["ndmi_p10"], ndmi_valid["ndmi_p90"],
                    alpha=0.15, color="steelblue")
    ax.set_ylabel("NDMI", fontsize=12)
else:
    ax.plot(ts_df["date"], ts_df["ndre_mean"], "o-", color="steelblue",
            linewidth=2, markersize=4, label="NDRE (moisture proxy)")
    ax.set_ylabel("NDRE (proxy)", fontsize=12)
ax.legend(loc="upper left", fontsize=10)
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---

## Summary (Part 2: Crop Monitoring)

### Pipeline Steps
1. **Scene discovery** -- auto-pair SR + UDM2 files by date
2. **AOI clipping** -- extract only the field of interest
3. **Cloud filtering** -- reject scenes below the clean-pixel threshold
4. **Index computation** -- NDVI, NDRE, EVI, SAVI per date
5. **Time-series assembly** -- field-level stats (mean, std, percentiles)
6. **Phenology curves** -- visualize growth trajectory
7. **Growth stage detection** -- automatic classification from NDVI shape
8. **Stress alerts** -- flag sudden drops, NDRE divergence, spatial variability
9. **NDMI fusion** -- Sentinel-2 SWIR for true moisture index
10. **Export** -- CSV time-series + JSON alerts for downstream use

### Stress Detection Signals
| Condition | Signal |
|-----------|--------|
| Water stress | NDRE drops while NDVI remains high |
| Nutrient deficiency | Lower NDVI plateau vs. reference |
| Pest / disease | Localized NDVI drop (high intra-field std) |
| Lodging | Sudden mid-season NDVI collapse |

### Key Advantages of PlanetScope for Crop Monitoring
- **3m resolution** -- field-level and sub-field variability mapping
- **Daily revisit** -- capture fast phenological transitions
- **Red Edge band** -- NDRE detects stress before NDVI shows visible decline
- **Fusion with Sentinel-2 SWIR** -- enables true NDMI at higher spatial resolution

---

# Part 3: Planet API Scene Ordering from a KML File

This section uses the **Planet SDK for Python** (synchronous `Planet()` client) to:
1. Parse a KML file into a GeoJSON geometry
2. Search the Planet Data API for available PlanetScope scenes
3. Place and download orders via the Orders API

**Prerequisites:**
- `pip install planet` (Planet SDK for Python v2+)
- A valid Planet API key with data access (see access options below)

**Product bundle:** `analytic_8b_sr_udm2` (8-band Surface Reflectance + UDM2 mask)

## 22. Part 3 Setup and Configuration

In [ ]:
# Install the Planet SDK if needed
# !pip install planet

import os

# ----- CONFIGURATION -----

# Planet API key: set via environment variable or paste directly
PL_API_KEY = os.environ.get("PL_API_KEY", "YOUR_API_KEY_HERE")

# Path to the KML file (already in the project folder)
KML_PATH = "IND-CG-600000_AMLIPARA ANIL KUMAR NETAM USRH-24 BALJI.kml"

# Date range for scene search (full growing season)
ORDER_START_DATE = "2024-06-01"
ORDER_END_DATE   = "2024-12-31"

# Maximum cloud cover (fraction, 0-1)
MAX_CLOUD_COVER = 0.15

# Dry-run mode: when True, searches but does NOT place real orders
DRY_RUN = True

# Download directory for ordered scenes
DOWNLOAD_DIR = "orders"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print(f"KML file : {KML_PATH}")
print(f"Date range : {ORDER_START_DATE} to {ORDER_END_DATE}")
print(f"Max cloud  : {MAX_CLOUD_COVER*100:.0f}%")
print(f"Dry run    : {DRY_RUN}")

## 23. Parse KML to GeoJSON

Extract the polygon geometry from a KML file using only the Python standard library.

In [ ]:
import xml.etree.ElementTree as ET
import json

def kml_to_geojson(kml_path):
    """Parse a KML file and return the first polygon as a GeoJSON geometry dict."""
    tree = ET.parse(kml_path)
    root = tree.getroot()

    # Handle KML namespace
    ns = ""
    if root.tag.startswith("{"):
        ns = root.tag.split("}")[0] + "}"

    # Find the first Polygon's coordinates
    coords_el = root.find(f".//{ns}Polygon//{ns}coordinates")
    if coords_el is None:
        raise ValueError(f"No Polygon found in {kml_path}")

    raw = coords_el.text.strip()
    ring = []
    for pt in raw.split():
        parts = pt.split(",")
        lon, lat = float(parts[0]), float(parts[1])
        ring.append([lon, lat])

    # Ensure the ring is closed
    if ring[0] != ring[-1]:
        ring.append(ring[0])

    geojson_geom = {"type": "Polygon", "coordinates": [ring]}

    # Print summary
    lons = [c[0] for c in ring]
    lats = [c[1] for c in ring]
    print(f"Parsed {len(ring)} vertices from KML")
    print(f"Bounding box: ({min(lons):.6f}, {min(lats):.6f}) to ({max(lons):.6f}, {max(lats):.6f})")

    return geojson_geom

# Parse the KML
aoi_geojson = kml_to_geojson(KML_PATH)

# Optionally save as GeoJSON for reuse
geojson_path = KML_PATH.rsplit(".", 1)[0] + ".geojson"
with open(geojson_path, "w") as f:
    json.dump({"type": "Feature", "geometry": aoi_geojson, "properties": {}}, f, indent=2)
print(f"Saved GeoJSON: {geojson_path}")

## 24. Search the Planet Data API

Use the synchronous `Planet()` client with `data_filter` to find PlanetScope scenes matching our criteria.

In [ ]:
from datetime import datetime
from planet import Planet
from planet import data_filter as filters

def search_scenes(api_key, geometry, start_date, end_date, max_cloud=0.15):
    """Search for PlanetScope 8-band scenes using the synchronous Planet client."""
    pl = Planet(api_key=api_key)

    # Build combined filter
    combined = filters.and_filter([
        filters.geometry_filter(geometry),
        filters.date_range_filter("acquired",
                                  gte=datetime.fromisoformat(start_date),
                                  lte=datetime.fromisoformat(end_date)),
        filters.range_filter("cloud_cover", lte=max_cloud),
        filters.permission_filter(),
    ])

    # Search -- returns an iterator
    items = []
    for item in pl.data.search(["PSScene"], combined, limit=500):
        items.append(item)

    print(f"Found {len(items)} scenes")
    return items

# Run the search
scenes = search_scenes(PL_API_KEY, aoi_geojson, ORDER_START_DATE, ORDER_END_DATE, MAX_CLOUD_COVER)

# Show summary table
if scenes:
    print(f"\n{'Date':20s} {'Cloud%':>7s} {'Clear%':>7s} {'ID'}")
    print("-" * 80)
    for s in scenes[:20]:  # Show first 20
        props = s["properties"]
        acq = props.get("acquired", "")[:19]
        cc = props.get("cloud_cover", 0) * 100
        clr = props.get("clear_percent", 0)
        print(f"{acq:20s} {cc:6.1f}% {clr:6.1f}% {s['id']}")
    if len(scenes) > 20:
        print(f"  ... and {len(scenes) - 20} more scenes")

## 25. Place Orders via the Orders API

Build order requests with clip and harmonize tools, then place them using the synchronous client.
- **Clip tool**: Crops each scene to the AOI polygon (saves bandwidth and storage)
- **Harmonize tool**: Normalizes across SuperDove sensor generations
- **Fallback bundle**: Falls back to 4-band if 8-band is unavailable for older scenes
- Max 500 items per order (API limit); batches automatically if needed

In [ ]:
from planet import Planet
from planet.order_request import build_request, product, clip_tool, harmonize_tool

def build_orders(item_ids, geometry, order_name_prefix="ps_order"):
    """Build one or more order request dicts. Splits into batches of 500 if needed."""
    BATCH_SIZE = 500
    orders = []

    for batch_num, i in enumerate(range(0, len(item_ids), BATCH_SIZE)):
        batch_ids = item_ids[i : i + BATCH_SIZE]
        name = f"{order_name_prefix}_batch{batch_num + 1}" if len(item_ids) > BATCH_SIZE else order_name_prefix

        request = build_request(
            name=name,
            products=[
                product(
                    item_ids=batch_ids,
                    product_bundle="analytic_8b_sr_udm2",
                    item_type="PSScene",
                    fallback_bundle="analytic_sr_udm2,analytic_udm2",
                )
            ],
            tools=[
                clip_tool(aoi=geometry),
                harmonize_tool(target_sensor="Sentinel-2A"),
            ],
        )
        orders.append(request)
        print(f"  Order '{name}': {len(batch_ids)} items")

    return orders


def place_and_download_orders(api_key, order_requests, download_dir, dry_run=True):
    """Place orders and optionally wait + download. Respects dry_run flag."""
    if dry_run:
        print(f"\n[DRY RUN] Would place {len(order_requests)} order(s):")
        for req in order_requests:
            n_items = sum(len(p["item_ids"]) for p in req["products"])
            print(f"  '{req['name']}': {n_items} items, bundle={req['products'][0]['product_bundle']}")
        print("\nSet DRY_RUN = False to place real orders.")
        return []

    pl = Planet(api_key=api_key)
    placed_orders = []

    for req in order_requests:
        print(f"\nPlacing order: {req['name']} ...")
        order = pl.orders.create_order(req)
        order_id = order["id"]
        print(f"  Order ID: {order_id}, state: {order['state']}")
        placed_orders.append(order)

        # Wait for the order to complete
        print(f"  Waiting for order {order_id} ...")
        pl.orders.wait(order_id, callback=lambda s: print(f"    state: {s}"))

        # Download
        print(f"  Downloading to {download_dir} ...")
        pl.orders.download_order(order_id, directory=download_dir, overwrite=True)
        print(f"  Download complete.")

    return placed_orders

## 26. Execute: Search and Order

In [ ]:
# Build the list of item IDs from our search results
item_ids = [s["id"] for s in scenes]
print(f"Total scenes to order: {len(item_ids)}")

if item_ids:
    # Build order request(s)
    order_requests = build_orders(item_ids, aoi_geojson, order_name_prefix="ps_vegetation_order")

    # Place orders (or dry-run preview)
    placed = place_and_download_orders(PL_API_KEY, order_requests, DOWNLOAD_DIR, dry_run=DRY_RUN)
else:
    print("No scenes found -- adjust date range or cloud cover threshold.")

## 27. Utility: List Existing Orders

In [ ]:
def list_recent_orders(api_key, limit=10):
    """List recent orders and their status."""
    pl = Planet(api_key=api_key)

    print(f"{'Order ID':40s} {'Name':30s} {'State':10s} {'Created'}")
    print("-" * 100)

    count = 0
    for order in pl.orders.list_orders():
        if count >= limit:
            break
        print(f"{order['id']:40s} {order.get('name',''):30s} {order['state']:10s} {order.get('created_on','')[:19]}")
        count += 1

    if count == 0:
        print("  No orders found.")

# Uncomment to list your recent orders:
# list_recent_orders(PL_API_KEY)

---

## Summary (Part 3)

### Workflow Recap
1. **Parse KML** to GeoJSON polygon (stdlib `xml.etree.ElementTree`)
2. **Search** the Planet Data API for PSScene items matching date range, cloud cover, and AOI
3. **Build orders** with clip + harmonize tools, batched at 500 items max
4. **Place and download** orders (with dry-run safety switch)

### Configuration Reference

| Parameter | Default | Description |
|-----------|---------|-------------|
| `PL_API_KEY` | env var | Your Planet API key |
| `KML_PATH` | *(pre-filled)* | Path to field boundary KML |
| `ORDER_START_DATE` | `2024-06-01` | Season start |
| `ORDER_END_DATE` | `2024-12-31` | Season end |
| `MAX_CLOUD_COVER` | `0.15` | Max cloud cover (0-1) |
| `DRY_RUN` | `True` | Safety switch: preview without ordering |
| `DOWNLOAD_DIR` | `orders/` | Where downloaded scenes are saved |

### API Notes
- Uses the **synchronous `Planet()` client** (simpler than the async `Session` API)
- `fallback_bundle` automatically falls back to 4-band SR if 8-band is unavailable
- `harmonize_tool` normalizes reflectance across different SuperDove sensor generations
- Orders API rate limit: ~5 concurrent orders, ~500 items per order
- Downloaded scenes feed directly into Parts 1 and 2 of this notebook

### Next Steps
- Set `DRY_RUN = False` and run to place real orders
- Point Part 2's `SCENES_DIR` at the `orders/` download directory
- Run the full crop monitoring pipeline on the downloaded time-series